# ICT-4202 | Artificial Intelligence and Machine Learning Laboratory
## Final Laboratory Examination — Set 06
**Islamic University, Kushtia, Bangladesh**  
Department of Information and Communication Technology

> **Notebook purpose:** Question-matched solutions with brief background theory, mathematical notation, readable Python, reproducible output, interpretation, and viva revision. This is a learning/reporting notebook, not a claim of official model answers.

| Question | Topic | Marks |
|:--|:--|--:|
| Q1 | Student eligibility using forward chaining | 25 |
| Q2 | BFS, DFS and IDDFS | 30 |
| Q3 | k-NN versus decision tree | 35 |
| Q4 | Viva: logic, implication, BFS/DFS/IDDFS, classification | 10 |

**Execution:** Open in JupyterLab/VS Code, choose a Python kernel, then use **Run All**. All random experiments use a fixed seed. Markdown cells use MathJax (`$...$` and `$$...$$`) to render equations on GitHub/Jupyter.

**Quick access - one file per numbered question:** [Q1 / A](Set_06_A.py) | [Q2 / B](Set_06_B.py) | [Q3 / C](Set_06_C.py) | [Q4 / D: Viva practice](Set_06_D.py).

Q1-Q3 contain the required practical work (including all subparts). Q4 is a viva, so its `.py` file is a revision/self-quiz tool, not an extra algorithm assignment. This notebook contains solutions, explanations, mathematical theory, and the Q4 viva questions.

---
## Q1. Propositional Logic (25 marks)
**Question specification**  
`Registered`; `Registered→Student`; `Student→HasID`; `Student AND PaidFee→ExamEligible`; `ExamEligible→CanEnterExamHall`. Infer for Registered alone and Registered + PaidFee, display traces, and construct implication truth table.

### Theory and mathematical background
A knowledge base (KB) stores **facts** and **rules**. In propositional forward chaining, rules have premises and a conclusion. For example:
$$P\land Q\Rightarrow R.$$
A rule fires when all its premises are already known. This is repeated until no new conclusion can be inferred (the *fixed point*).

For implication, $P\Rightarrow Q\equiv \neg P\lor Q$; it is false only when $P$ is true and $Q$ is false. Forward chaining for definite Horn-style rules is a data-driven form of logical inference.

In [1]:

def forward_chain(initial_facts, rules):
    """Data-driven inference: rules are (premise-set, conclusion) pairs."""
    known = set(initial_facts)
    trace = []
    while True:
        newly_added = False
        for premises, conclusion in rules:
            if premises.issubset(known) and conclusion not in known:
                known.add(conclusion)
                trace.append((sorted(premises), conclusion))
                newly_added = True
        if not newly_added:
            break
    return known, trace

def print_inference(label, initial, rules):
    known, trace = forward_chain(initial, rules)
    print(f"\n{label} | Initial facts: {sorted(initial)}")
    for i, (prem, conclusion) in enumerate(trace, 1):
        print(f"  {i}. {prem}  =>  {conclusion}")
    if not trace: print("  No new rule fired.")
    print('Final facts:', sorted(known))
    return known, trace

rules = [({'Registered'},'Student'),({'Student'},'HasID'),
         ({'Student','PaidFee'},'ExamEligible'),
         ({'ExamEligible'},'CanEnterExamHall')]
facts1,trace1=print_inference('Case 1: Registered only',{'Registered'},rules)
facts2,trace2=print_inference('Case 2: Registered + PaidFee',{'Registered','PaidFee'},rules)
assert 'CanEnterExamHall' not in facts1 and 'CanEnterExamHall' in facts2

import itertools
print('  P      Q      P -> Q')
for p, q in itertools.product([False, True], repeat=2):
    implication = (not p) or q
    print(f'{str(p):5}  {str(q):5}  {implication}')



Case 1: Registered only | Initial facts: ['Registered']
  1. ['Registered']  =>  Student
  2. ['Student']  =>  HasID
Final facts: ['HasID', 'Registered', 'Student']

Case 2: Registered + PaidFee | Initial facts: ['PaidFee', 'Registered']
  1. ['Registered']  =>  Student
  2. ['Student']  =>  HasID
  3. ['PaidFee', 'Student']  =>  ExamEligible
  4. ['ExamEligible']  =>  CanEnterExamHall
Final facts: ['CanEnterExamHall', 'ExamEligible', 'HasID', 'PaidFee', 'Registered', 'Student']
  P      Q      P -> Q
False  False  True
False  True   True
True   False  False
True   True   True


### Truth table: implication $P\Rightarrow Q$  
Recall: only **True → False** makes the implication false.

### Result interpretation and discussion
Eligibility is not inferred until the **conjunction** of `Student` and `PaidFee` is satisfied. That rule adds `ExamEligible`, enabling the downstream `CanEnterExamHall` conclusion. Forward chaining stops once a complete scan produces no additional fact.

---
## Q2. BFS, DFS and IDDFS (30 marks)
**Question specification**  
`S→{A,B}, A→{C,D}, B→{E,F}, C→{G}, D→{H}, E→{I}, F→{J}`. Search S to G using BFS, DFS and IDDFS. Show paths, nodes expanded, and memory behavior.

### Theory and mathematical background
For an unweighted graph, **BFS** expands the shallowest frontier state first (FIFO queue), so it finds a minimum-edge path when each edge has equal cost. **DFS** explores a branch deeply first (LIFO stack), so its outcome depends on neighbor order. A **visited set** prevents re-expansion.

With $V$ vertices and $E$ edges, BFS and DFS can both run in $O(V+E)$ time on an adjacency-list graph; their frontier sizes and discovered-node storage differ. In tree-search notation, BFS typically uses $O(b^{d+1})$ space, while DFS uses $O(bm)$ space ($b$ branching factor, $d$ shallowest goal depth, $m$ maximum depth). These are different models of complexity.

**Depth-Limited Search (DLS)** stops descending beyond an allowed depth $\ell$. A depth cutoff means the search might succeed with a larger limit; *failure* means no answer was found without such a cutoff.

**Iterative Deepening DFS (IDDFS)** performs DLS repeatedly at limits $0,1,2,\ldots$ until a goal is found:
$$IDDFS(\ell)=DLS(0),DLS(1),\dots,DLS(\ell).$$
It revisits shallow nodes, trading some repeated time for the memory efficiency of depth-first search. In a tree, DLS is commonly characterized by $O(b^\ell)$ time and $O(b\ell)$ space when sibling iterators are retained; IDDFS has roughly $O(b^d)$ time and $O(bd)$ space for a finite branching factor.

In [2]:
from collections import deque

import heapq

from itertools import count

def bfs(graph, start, goal):
    queue = deque([(start, [start])])
    seen = {start}
    expanded = []
    while queue:
        node, path = queue.popleft()
        expanded.append(node)
        if node == goal:
            return path, expanded
        for nxt in graph.get(node, []):
            if nxt not in seen:
                seen.add(nxt)
                queue.append((nxt, path + [nxt]))
    return None, expanded

def dfs(graph, start, goal):
    stack = [(start, [start])]
    seen = set()
    expanded = []
    while stack:
        node, path = stack.pop()
        if node in seen: continue
        seen.add(node)
        expanded.append(node)
        if node == goal:
            return path, expanded
        # Reverse push order so the FIRST listed neighbor is explored first.
        for nxt in reversed(graph.get(node, [])):
            if nxt not in seen:
                stack.append((nxt, path + [nxt]))
    return None, expanded

def depth_limited_search(graph, start, goal, limit):
    """Return path, status (success/cutoff/failure), expanded sequence."""
    expanded = []
    def visit(node, path, depth):
        expanded.append((node, depth))
        if node == goal: return path, 'success'
        if depth >= limit:
            eligible = [v for v in graph.get(node, []) if v not in path]
            return None, ('cutoff' if eligible else 'failure')
        had_cutoff = False
        for nxt in graph.get(node, []):
            if nxt in path: continue
            result, status = visit(nxt, path + [nxt], depth + 1)
            if status == 'success': return result, status
            if status == 'cutoff': had_cutoff = True
        return None, 'cutoff' if had_cutoff else 'failure'
    path, status = visit(start, [start], 0)
    return path, status, expanded

def iddfs(graph, start, goal, max_depth=10):
    iterations = []
    for limit in range(max_depth+1):
        path, status, expanded = depth_limited_search(graph, start, goal, limit)
        iterations.append((limit, path, status, expanded))
        if status == 'success': break
        if status == 'failure': break
    return path, iterations

def print_iddfs(graph, start, goal, max_depth=10):
    path, iterations = iddfs(graph, start, goal, max_depth)
    for limit, candidate, status, expanded in iterations:
        trace = ' -> '.join(node for node, depth in expanded)
        print(f'Limit={limit} | status={status:7} | expanded={trace} | path={candidate}')
    print(f'IDDFS path: {path}; root visits: {len(iterations)}')
    return path, iterations

graph={'S':['A','B'],'A':['C','D'],'B':['E','F'],'C':['G'],
       'D':['H'],'E':['I'],'F':['J'],'G':[],'H':[],'I':[],'J':[]}
for name, fn in [('BFS',bfs),('DFS',dfs)]:
    path,trace=fn(graph,'S','G')
    print(f'{name}: path={path}; expanded={trace}; count={len(trace)}')
path,iterations=print_iddfs(graph,'S','G',8)
assert path==['S','A','C','G']


BFS: path=['S', 'A', 'C', 'G']; expanded=['S', 'A', 'B', 'C', 'D', 'E', 'F', 'G']; count=8
DFS: path=['S', 'A', 'C', 'G']; expanded=['S', 'A', 'C', 'G']; count=4
Limit=0 | status=cutoff  | expanded=S | path=None
Limit=1 | status=cutoff  | expanded=S -> A -> B | path=None
Limit=2 | status=cutoff  | expanded=S -> A -> C -> D -> B -> E -> F | path=None
Limit=3 | status=success | expanded=S -> A -> C -> G | path=['S', 'A', 'C', 'G']
IDDFS path: ['S', 'A', 'C', 'G']; root visits: 4


### Result interpretation and discussion
The goal is reached at depth 3. BFS uses a queue and stores potentially broad frontiers. DFS stores a narrower depth-oriented frontier but can wander deeply. IDDFS repeats the shallow levels but returns a shallowest-depth solution under finite branching.

---
## Q3. k-NN vs Decision Tree (35 marks)
**Question specification**  
Same stratified 70/30 Iris split for k-NN $k=5$ and a decision tree `max_depth=3`. Print both accuracies/confusion matrices, identify incorrect samples, standardize and repeat k-NN.

### Theory and mathematical background
**k-Nearest Neighbors (k-NN)** predicts the majority label among the $k$ nearest training samples. A common distance is Euclidean:
$$d(\mathbf x,\mathbf z)=\sqrt{\sum_{j=1}^{p}(x_j-z_j)^2}.$$
Standardization rescales a feature using training-set statistics:
$$z_j=\frac{x_j-\mu_j^{\mathrm{train}}}{\sigma_j^{\mathrm{train}}}.$$
**Accuracy** $=(TP+TN)/(TP+TN+FP+FN)$ for binary labels (or total correct / total cases for multiclass). A confusion matrix $C_{ij}$ counts instances of actual class $i$ predicted as class $j$. Fit scaling on training samples only to prevent leakage.

A **decision tree** recursively chooses feature thresholds that reduce node impurity. One measure is Gini impurity:
$$Gini(D)=1-\sum_{c=1}^{K}p(c\mid D)^2.$$
Restricting `max_depth` reduces complexity but can cause underfitting; large or unrestricted depth can memorize training samples and overfit.

For each class, $Precision=\frac{TP}{TP+FP}$, $Recall=\frac{TP}{TP+FN}$, and $F_1=\frac{2PR}{P+R}$. Macro averages assign equal weight to every class.

In [3]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import pandas as pd
import numpy as np
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,
                                                    stratify=y, random_state=42)
print('Iris samples:', X.shape, '| train:', len(y_train), '| test:', len(y_test))
print('Class order:', list(iris.target_names))

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train,y_train)
dt = DecisionTreeClassifier(max_depth=3,random_state=42).fit(X_train,y_train)
for name, model in [('k-NN (k=5)',knn),('Decision tree (depth=3)',dt)]:
    preds=model.predict(X_test)
    errors=np.flatnonzero(preds!=y_test)
    print(f'\n{name}: accuracy={accuracy_score(y_test,preds):.4f}')
    print('Confusion matrix (actual rows):\n',confusion_matrix(y_test,preds))
    print('Misclassified TEST ROW POSITIONS:',errors.tolist())
    for idx in errors[:10]:
        print(' ',idx, 'true:',iris.target_names[y_test[idx]],
              'predicted:',iris.target_names[preds[idx]],
              'features:',np.round(X_test[idx],2).tolist())
scaled_knn=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5))
scaled_knn.fit(X_train,y_train)
scaled_pred=scaled_knn.predict(X_test)
print('\nScaled k-NN accuracy:',round(accuracy_score(y_test,scaled_pred),4))
print('Scaled confusion matrix:\n',confusion_matrix(y_test,scaled_pred))


Iris samples: (150, 4) | train: 105 | test: 45
Class order: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]

k-NN (k=5): accuracy=0.9778
Confusion matrix (actual rows):
 [[15  0  0]
 [ 0 15  0]
 [ 0  1 14]]
Misclassified TEST ROW POSITIONS: [42]
  42 true: virginica predicted: versicolor features: [4.9, 2.5, 4.5, 1.7]

Decision tree (depth=3): accuracy=0.9778
Confusion matrix (actual rows):
 [[15  0  0]
 [ 0 14  1]
 [ 0  0 15]]
Misclassified TEST ROW POSITIONS: [39]
  39 true: versicolor predicted: virginica features: [6.7, 3.0, 5.0, 1.7]

Scaled k-NN accuracy: 0.9111
Scaled confusion matrix:
 [[15  0  0]
 [ 0 15  0]
 [ 0  4 11]]


### Result interpretation and discussion
Distance-based k-NN is sensitive to feature units because feature scaling changes Euclidean distances and potentially neighbor ranks. Decision trees split on individual feature thresholds and are mostly invariant to positive linear rescaling. The train/test split remains identical for fair comparison.

---
## Q4. Viva voce (10 marks)

| Question | Concise answer |
|:--|:--|
| Forward chaining? | Infer consequences repeatedly from satisfied premises. |
| When is P→Q false? | When P is true but Q is false. |
| What is a BFS frontier? | Queue of discovered paths/nodes awaiting expansion. |
| Why IDDFS uses less memory than BFS? | It performs depth-limited DFS rather than storing a full breadth frontier. |
| How k-NN works? | Votes among the k closest training instances. |
| Why standardize k-NN inputs? | To reduce unit-driven dominance in distances. |
| Decision tree? | Predicts via learned nested feature thresholds. |

| Which data structure does DFS use? | A LIFO stack, or the program call stack in recursive DFS. |
| How does IDDFS combine BFS and DFS benefits? | It finds a shallow goal by trying larger limits while retaining depth-first memory use. |

**Review tip:** Explain the code by identifying the inputs, data structures, stopping conditions, evaluation metrics, and one limitation for each task.

---
### Submission checklist
- [ ] Execute every Python cell in sequence; check the saved output is current.
- [ ] Explain the main formula, trace, and interpretation for Q1, Q2, Q3.
- [ ] Record the original task settings, dataset, random seed, and actual test metrics.
- [ ] For WEKA tasks, run WEKA locally and attach the real result-buffer output/screenshot; notebook instructions **do not substitute for WEKA evidence**.
- [ ] Be ready to discuss one methodological limitation in viva.

**Integrity note:** Code and explanations are independently written for this study repository. Numeric results are calculated from the shown code and may vary with library versions where numerical or tie-breaking differences occur.